# Reto completo: del dataset al despliegue de Machine Learning
## Ruta A – Aprendizaje supervisado (clasificación)
**Google Colab + datos propios + Streamlit + GitHub + Docker + Google Cloud Compute Engine**

**Autores (pareja):** Julián Jiménez · Tomás Rincón

Este notebook es **el único entregable de desarrollo**: al ejecutarlo de principio a fin
(`Entorno de ejecución → Ejecutar todo`) genera todo el proyecto en `/content/ml_streamlit_project`:

| Archivo | Rol |
|---|---|
| `data/datos.csv` | Dataset (Customer Churn Iran, repo del curso) |
| `ml_core.py` | Lógica común: inferencia de variables, Pipeline, evaluación, artefacto, predicción |
| `train.py` | Entrenamiento reproducible → `model.joblib` |
| `model.joblib` | Artefacto: pipeline + metadatos (features, categorías, rangos, métricas) |
| `app.py` | Streamlit: **entrenar con cualquier CSV**, predecir un caso, predecir por lotes |
| `requirements.txt`, `Dockerfile`, `.dockerignore`, `.streamlit/config.toml` | Empaquetado |
| `README.md` | Instrucciones completas de reproducción y despliegue |

Flujo (PDF §3): dataset → exploración → `train.py` → entrenar/evaluar → `model.joblib` → `app.py` →
probar Streamlit en Colab → archivos Docker → GitHub → VM GCP → Docker → `git clone` → `docker build` →
`docker run` → firewall → IP pública → prueba cruzada.

## 4. Preparación del proyecto en Google Colab
### Paso 4.1 – Instalar dependencias

In [ ]:
!pip -q install streamlit scikit-learn pandas numpy joblib openpyxl

### Paso 4.2 – Crear la carpeta del proyecto

In [ ]:
from pathlib import Path
import os, sys

PROJECT = Path("/content/ml_streamlit_project")
PROJECT.mkdir(exist_ok=True)
(PROJECT / "data").mkdir(exist_ok=True)
(PROJECT / ".streamlit").mkdir(exist_ok=True)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print(PROJECT)

### Paso 4.3 – Recordatorio sobre persistencia
El almacenamiento de Colab es **temporal**: un reinicio del runtime puede borrar los archivos.
Al final del notebook se genera `ml_streamlit_project.zip` para descargarlo y subirlo a GitHub.

## Configuración del experimento
Cambiando **solo esta celda** el proyecto completo se reentrena con otro CSV de clasificación
(por ejemplo `Default_Clientes.csv` del repo, objetivo `Default`, que usa `;` como separador).

In [ ]:
DATA_URL = ("https://raw.githubusercontent.com/wilmerpineda/machine_learning_ii_course_public/"
            "main/datasets/public/Customer_Churn_Iran.csv")
TARGET = "Churn"
EXTRA_DROP = ["Customer Value", "Age"]   # Customer Value: fuga; Age: redundante con Age Group
MODEL_NAME = "random_forest"      # "logistic_regression" o "random_forest"
USE_UPLOAD = False                # True -> seleccionar un CSV/Excel propio desde el computador
RANDOM_STATE = 42

## Módulo común `ml_core.py`
Contiene el Pipeline del PDF §7.3 (imputación + escalado / imputación + OneHot + modelo),
la inferencia automática de variables numéricas/categóricas, la evaluación del PDF §8.1 y la
construcción del artefacto (PDF §8.2 y §14). Lo usamos **tanto en `train.py` como en `app.py`**, por lo que
las transformaciones de entrenamiento y de serving son idénticas.

In [ ]:
%%writefile /content/ml_streamlit_project/ml_core.py
"""ml_core.py - Lógica común de entrenamiento e inferencia (Ruta A - supervisada).

La usan train.py (entrenamiento reproducible) y app.py (Streamlit), de modo que
las transformaciones de entrenamiento y de serving sean SIEMPRE las mismas
(Pipeline de scikit-learn: preprocesamiento + modelo en un solo objeto).
"""
from __future__ import annotations

import io
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

RANDOM_STATE = 42
MAX_CAT_UNIQUE = 10        # enteros con <= 10 valores distintos se tratan como categóricos
MAX_CLASSES = 20           # más clases que esto -> probablemente es regresión
MODELS = {
    "logistic_regression": "Regresión logística",
    "random_forest": "Random Forest",
}


# ----------------------------------------------------------------------------
# Lectura y limpieza
# ----------------------------------------------------------------------------
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normaliza nombres: quita BOM y espacios repetidos ('Call  Failure' -> 'Call Failure')."""
    df = df.copy()
    df.columns = [" ".join(str(c).replace("\ufeff", "").split()) for c in df.columns]
    return df


def read_csv_auto(source) -> pd.DataFrame:
    """Lee un CSV desde ruta, URL o archivo subido detectando el separador (',' ';' tab '|')."""
    if hasattr(source, "read"):  # archivo subido (Streamlit / Colab)
        raw = source.read()
        if isinstance(raw, bytes):
            raw = raw.decode("utf-8-sig", errors="replace")
        return clean_columns(pd.read_csv(io.StringIO(raw), sep=None, engine="python"))
    return clean_columns(pd.read_csv(source, sep=None, engine="python", encoding="utf-8-sig"))


def to_str(X):
    """Convierte categóricas a texto para que 1 y '1' sean la misma categoría."""
    return pd.DataFrame(X).astype(str).values


# ----------------------------------------------------------------------------
# Inferencia automática de variables
# ----------------------------------------------------------------------------
def infer_feature_types(X: pd.DataFrame, max_cat_unique: int = MAX_CAT_UNIQUE):
    numeric, categorical = [], []
    for col in X.columns:
        s = X[col]
        if pd.api.types.is_bool_dtype(s) or not pd.api.types.is_numeric_dtype(s):
            categorical.append(col)
        elif pd.api.types.is_integer_dtype(s) and s.nunique(dropna=True) <= max_cat_unique:
            categorical.append(col)
        else:
            numeric.append(col)
    return numeric, categorical


def suggest_drop_columns(df: pd.DataFrame, target: str) -> list[str]:
    """Sugiere excluir identificadores, constantes y texto libre de alta cardinalidad."""
    n = len(df)
    drop = []
    for col in df.columns:
        if col == target:
            continue
        s = df[col]
        nunique = s.nunique(dropna=True)
        name = col.lower()
        is_id_name = name in {"id", "index", "instant"} or name.endswith("_id") or name.startswith("id_")
        if nunique <= 1:
            drop.append(col)
        elif is_id_name:
            drop.append(col)
        elif nunique == n and not pd.api.types.is_float_dtype(s):
            drop.append(col)
        elif s.dtype == object and nunique > 0.5 * n:
            drop.append(col)
    return drop


# ----------------------------------------------------------------------------
# Pipeline (PDF §7.3)
# ----------------------------------------------------------------------------
def build_pipeline(numeric_features, categorical_features, model_name="random_forest") -> Pipeline:
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("to_str", FunctionTransformer(to_str)),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features),
    ])
    if model_name == "logistic_regression":
        classifier = LogisticRegression(max_iter=1000)
    elif model_name == "random_forest":
        # Configuración tomada del notebook de semana 14 del curso
        classifier = RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    else:
        raise ValueError(f"Modelo no soportado: {model_name}")
    return Pipeline([("preprocessor", preprocessor), ("classifier", classifier)])


# ----------------------------------------------------------------------------
# Entrenamiento + evaluación + artefacto (PDF §7, §8, §14)
# ----------------------------------------------------------------------------
def validate_target(y: pd.Series) -> None:
    n_classes = y.nunique()
    if n_classes < 2:
        raise ValueError("La variable objetivo debe tener al menos 2 clases.")
    if n_classes > MAX_CLASSES:
        raise ValueError(
            f"La variable objetivo tiene {n_classes} valores distintos: parece un problema de "
            "regresión. Esta aplicación implementa la ruta supervisada de CLASIFICACIÓN."
        )


def evaluate(model, X_test, y_test) -> dict:
    pred = model.predict(X_test)
    classes = list(model.classes_)
    metrics = {
        "accuracy": float(accuracy_score(y_test, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_test, pred)),
        "f1_weighted": float(f1_score(y_test, pred, average="weighted")),
        "recall_macro": float(recall_score(y_test, pred, average="macro")),
    }
    proba = model.predict_proba(X_test)
    try:
        if len(classes) == 2:
            pos = classes[1]
            metrics["roc_auc"] = float(roc_auc_score(y_test, proba[:, 1]))
            metrics["precision_pos"] = float(precision_score(y_test, pred, pos_label=pos, zero_division=0))
            metrics["recall_pos"] = float(recall_score(y_test, pred, pos_label=pos, zero_division=0))
            metrics["f1_pos"] = float(f1_score(y_test, pred, pos_label=pos, zero_division=0))
            metrics["average_precision"] = float(average_precision_score(y_test, proba[:, 1]))
        else:
            metrics["roc_auc_ovr"] = float(roc_auc_score(y_test, proba, multi_class="ovr", labels=classes))
    except ValueError:
        pass
    return {
        "metrics": metrics,
        "confusion_matrix": confusion_matrix(y_test, pred, labels=classes).tolist(),
        "classification_report": classification_report(y_test, pred, zero_division=0),
        "labels": [str(c) for c in classes],
    }


def cross_val_compare(
    df: pd.DataFrame,
    target: str,
    drop_columns=None,
    models=None,
    cv: int = 5,
    random_state: int = RANDOM_STATE,
) -> dict:
    """Validación cruzada estratificada sobre el dataset completo para comparar modelos.

    Se usa para elegir el modelo sin consumir el conjunto de prueba (evita el sesgo de
    elegir "el mejor" sobre el propio test). Devuelve media ± desviación de f1 (clase
    positiva), recall (clase positiva) y ROC-AUC para cada modelo.
    """
    df = clean_columns(df)
    target = " ".join(str(target).split())
    drop_columns = [" ".join(str(c).split()) for c in (drop_columns or [])]
    df = df.dropna(subset=[target])

    y = df[target]
    if y.dtype == object or pd.api.types.is_bool_dtype(y):
        y = y.astype(str)
    features = [c for c in df.columns if c != target and c not in drop_columns]
    X = df[features].copy()
    numeric_features, categorical_features = infer_feature_types(X)
    for col in categorical_features:
        X[col] = X[col].where(X[col].isna(), X[col].astype(str)).astype(object)

    models = list(models) if models else list(MODELS)
    pos = sorted(y.unique().tolist())[-1]
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)
    results = {}
    for name in models:
        f1s, recs, aucs = [], [], []
        for tr, te in skf.split(X, y):
            pipe = build_pipeline(numeric_features, categorical_features, name)
            pipe.fit(X.iloc[tr], y.iloc[tr])
            pred = pipe.predict(X.iloc[te])
            f1s.append(f1_score(y.iloc[te], pred, pos_label=pos, zero_division=0))
            recs.append(recall_score(y.iloc[te], pred, pos_label=pos, zero_division=0))
            try:
                aucs.append(roc_auc_score(y.iloc[te], pipe.predict_proba(X.iloc[te])[:, 1]))
            except (ValueError, IndexError):
                pass
        results[name] = {
            "f1_pos": (float(np.mean(f1s)), float(np.std(f1s))),
            "recall_pos": (float(np.mean(recs)), float(np.std(recs))),
            "roc_auc": (float(np.mean(aucs)), float(np.std(aucs))) if aucs else (float("nan"), float("nan")),
        }
    return results


def train(
    df: pd.DataFrame,
    target: str,
    drop_columns=None,
    model_name: str = "random_forest",
    test_size: float = 0.20,
    random_state: int = RANDOM_STATE,
    max_rows: int | None = None,
    dataset_name: str = "dataset.csv",
) -> dict:
    """Entrena el pipeline completo y devuelve el artefacto (dict) listo para joblib."""
    df = clean_columns(df)
    target = " ".join(str(target).split())
    if target not in df.columns:
        raise ValueError(f"La columna objetivo '{target}' no existe en el dataset.")

    drop_columns = [" ".join(str(c).split()) for c in (drop_columns or [])]
    df = df.dropna(subset=[target])
    if max_rows and len(df) > max_rows:
        df = df.sample(max_rows, random_state=random_state)

    y = df[target]
    if y.dtype == object or pd.api.types.is_bool_dtype(y):
        y = y.astype(str)
    validate_target(y)

    features = [c for c in df.columns if c != target and c not in drop_columns]
    if not features:
        raise ValueError("No quedan variables explicativas después de excluir columnas.")
    X = df[features].copy()
    numeric_features, categorical_features = infer_feature_types(X)
    for col in categorical_features:  # categorías como texto (mismo tratamiento que en serving)
        X[col] = X[col].where(X[col].isna(), X[col].astype(str)).astype(object)

    counts = y.value_counts()
    stratify = y if counts.min() >= 2 else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=stratify
    )

    model = build_pipeline(numeric_features, categorical_features, model_name)
    model.fit(X_train, y_train)
    evaluation = evaluate(model, X_test, y_test)

    category_options = {
        col: sorted(X_train[col].dropna().astype(str).unique().tolist())
        for col in categorical_features
    }
    numeric_stats = {
        col: {
            "min": float(X_train[col].min()),
            "max": float(X_train[col].max()),
            "median": float(X_train[col].median()),
            "integer": bool(pd.api.types.is_integer_dtype(X_train[col])),
        }
        for col in numeric_features
    }
    # Caso de referencia para verificar paridad entre entornos (patrón semana 14)
    ref_input = X_test.iloc[[0]]
    reference = {
        "input": {k: (None if pd.isna(v) else (v.item() if hasattr(v, "item") else v))
                  for k, v in ref_input.iloc[0].to_dict().items()},
        "proba": model.predict_proba(ref_input)[0].tolist(),
        "tolerance": 1e-9,
    }

    return {
        "model": model,
        "features": features,
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "task": "classification",
        "target": target,
        "classes": [c.item() if hasattr(c, "item") else c for c in model.classes_],
        "category_options": category_options,
        "numeric_stats": numeric_stats,
        "dropped_columns": drop_columns,
        "model_name": model_name,
        "metrics": evaluation["metrics"],
        "evaluation": evaluation,
        "dataset": dataset_name,
        "n_rows_train": int(len(X_train)),
        "n_rows_test": int(len(X_test)),
        "sklearn_version": sklearn.__version__,
        "trained_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "reference": reference,
    }


# ----------------------------------------------------------------------------
# Inferencia
# ----------------------------------------------------------------------------
def prepare_input(df: pd.DataFrame, artifact: dict):
    """Alinea un DataFrame nuevo con el contrato del artefacto.

    Devuelve (X, faltantes, sobrantes). Las columnas faltantes se crean vacías
    (el imputador del pipeline las completa) y las sobrantes se ignoran.
    """
    df = clean_columns(df)
    features = artifact["features"]
    missing = [c for c in features if c not in df.columns]
    extra = [c for c in df.columns if c not in features]
    X = df.reindex(columns=features)
    for col in artifact["numeric_features"]:
        X[col] = pd.to_numeric(X[col], errors="coerce")
    for col in artifact["categorical_features"]:
        X[col] = X[col].where(X[col].isna(), X[col].astype(str)).astype(object)
    return X, missing, extra


def predict_df(artifact: dict, df: pd.DataFrame):
    X, missing, extra = prepare_input(df, artifact)
    model = artifact["model"]
    out = df.copy()
    out["prediccion"] = model.predict(X)
    proba = model.predict_proba(X)
    for i, cls in enumerate(model.classes_):
        out[f"prob_{cls}"] = np.round(proba[:, i], 4)
    return out, missing, extra


def check_parity(artifact: dict) -> float:
    """Recalcula el caso de referencia; devuelve la diferencia máxima absoluta."""
    ref = artifact["reference"]
    X = pd.DataFrame([ref["input"]], columns=artifact["features"])
    proba = artifact["model"].predict_proba(X)[0]
    return float(np.max(np.abs(proba - np.array(ref["proba"]))))

## 5. Cargar los datos
Opción B (URL pública, por defecto) u Opción A (subir archivo propio).

In [ ]:
import pandas as pd
import ml_core

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    filename = next(iter(uploaded))
    print("Archivo cargado:", filename)
    if filename.lower().endswith(".csv"):
        df = ml_core.read_csv_auto(filename)
    elif filename.lower().endswith((".xlsx", ".xls")):
        df = ml_core.clean_columns(pd.read_excel(filename))
    else:
        raise ValueError("Use un archivo CSV o Excel.")
else:
    df = ml_core.read_csv_auto(DATA_URL)

df.head()

## 6. Revisión mínima del dataset
- **¿Qué representa una fila?** Un cliente de una compañía de telecomunicaciones iraní, con su uso agregado de 12 meses.
- **Objetivo:** predecir `Churn` (1 = el cliente abandona).
- **¿Qué variables pueden entrar?** Las de uso y perfil. Excluimos `Customer Value` porque se calcula a partir del uso (riesgo de fuga) y no está disponible para un cliente nuevo.
- Los nombres con doble espacio (`Call  Failure`) los normalizamos con `ml_core.clean_columns`.

In [ ]:
print(df.shape)
display(df.head())
display(df.dtypes)
display(df.isna().sum().sort_values(ascending=False).head(15))
display(df.describe(include="all").T)
print("Duplicados:", df.duplicated().sum())
display(df[TARGET].value_counts(normalize=True).rename("proporción"))

## 7. Ruta A – Construcción supervisada
### Paso 7.1 – Definir objetivo y variables (inferencia automática)

In [ ]:
drop_columns = ml_core.suggest_drop_columns(df, TARGET)
drop_columns += [c for c in EXTRA_DROP if c in df.columns and c not in drop_columns]

features = [c for c in df.columns if c not in drop_columns + [TARGET]]
X = df[features].copy()
y = df[TARGET].copy()
numeric_features, categorical_features = ml_core.infer_feature_types(X)

print("Excluidas:", drop_columns)
print("Numéricas:", numeric_features)
print("Categóricas:", categorical_features)

### Paso 7.2 – Separar entrenamiento y prueba

In [ ]:
from sklearn.model_selection import train_test_split

for col in categorical_features:
    X[col] = X[col].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(X_train.shape, X_test.shape)

### Paso 7.3 – Preprocesamiento + modelo (Pipeline)
Modelo base del PDF: regresión logística. Lo comparamos con Random Forest (configuración del notebook
de semana 14 del curso: 300 árboles, `min_samples_leaf=5`, `class_weight="balanced"`).

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score, roc_auc_score

results = {}
for name in ml_core.MODELS:
    model = ml_core.build_pipeline(numeric_features, categorical_features, name)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results[name] = {
        "f1_weighted": f1_score(y_test, pred, average="weighted"),
        "recall_clase_1": recall_score(y_test, pred, pos_label=model.classes_[-1]),
        "roc_auc": roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]) if len(model.classes_) == 2 else None,
    }
    print(f"===== {ml_core.MODELS[name]} =====")
    print(confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred))

pd.DataFrame(results).T.round(4)

## 8. Evaluación y elección del modelo
Las clases están desbalanceadas (≈16 % de churn), por eso *accuracy* no basta (PDF §8.1).
Elegimos el modelo con mejor **recall de la clase churn** y **F1 ponderado**: no detectar un cliente que
se va (falso negativo) cuesta más que contactar a uno que se habría quedado. ROC-AUC sirve de apoyo
porque no depende del umbral.

In [ ]:
best = max(results, key=lambda k: (results[k]["recall_clase_1"], results[k]["f1_weighted"]))
print("Modelo elegido:", ml_core.MODELS[best])
if best != MODEL_NAME:
    print(f"Nota: MODEL_NAME está configurado como '{MODEL_NAME}'; train.py usará el elegido -> '{best}'")
MODEL_NAME = best

### Paso 8.2 – Guardar artefacto (modelo + metadatos + `category_options`)

In [ ]:
import joblib

artifact = ml_core.train(df, TARGET, drop_columns, MODEL_NAME, random_state=RANDOM_STATE,
                         dataset_name="datos.csv")
joblib.dump(artifact, PROJECT / "model.joblib")
print({k: round(v, 4) for k, v in artifact["metrics"].items()})
print("Claves del artefacto:", list(artifact))

## 11–12. Script reproducible `train.py` y dataset dentro del proyecto

In [ ]:
df.to_csv(PROJECT / "data" / "datos.csv", index=False)
print("Dataset guardado en", PROJECT / "data" / "datos.csv", df.shape)

In [ ]:
%%writefile /content/ml_streamlit_project/train.py
"""train.py - Entrenamiento reproducible (PDF §11).

Uso:
    python train.py                                   # dataset por defecto (Churn Iran)
    python train.py --data otro.csv --target clase    # cualquier CSV de clasificación
    python train.py --drop "ID" "Otra columna" --model logistic_regression
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import joblib
import pandas as pd

import ml_core

DATA_PATH = Path("data/datos.csv")
DATA_URL = (
    "https://raw.githubusercontent.com/wilmerpineda/machine_learning_ii_course_public/"
    "main/datasets/public/Customer_Churn_Iran.csv"
)
MODEL_PATH = Path("model.joblib")
DEFAULT_TARGET = "Churn"
# "Customer Value" se deriva del uso (fuga de información); "Age" es redundante
# con "Age Group" (correspondencia 1:1), por lo que se excluye para evitar
# duplicar las mismas columnas one-hot.
DEFAULT_DROP = ["Customer Value", "Age"]


def load_data(path: Path) -> pd.DataFrame:
    if path.exists():
        return ml_core.read_csv_auto(path)
    print(f"{path} no existe; descargando desde {DATA_URL}")
    return ml_core.read_csv_auto(DATA_URL)


def train_model(df: pd.DataFrame, target: str, drop: list[str], model_name: str, name: str) -> dict:
    artifact = ml_core.train(df, target=target, drop_columns=drop, model_name=model_name, dataset_name=name)
    ev = artifact["evaluation"]
    print("Variables numéricas:", artifact["numeric_features"])
    print("Variables categóricas:", artifact["categorical_features"])
    print("Matriz de confusión (filas=real, columnas=predicho):", ev["labels"])
    for row in ev["confusion_matrix"]:
        print("   ", row)
    print(ev["classification_report"])
    print(json.dumps({k: round(v, 4) for k, v in artifact["metrics"].items()}, indent=2))
    return artifact


def main() -> None:
    parser = argparse.ArgumentParser(description="Entrena el pipeline supervisado y guarda model.joblib")
    parser.add_argument("--data", type=Path, default=DATA_PATH)
    parser.add_argument("--target", default=DEFAULT_TARGET)
    parser.add_argument("--drop", nargs="*", default=None,
                        help="Columnas a excluir (por defecto: sugeridas + Customer Value si existe)")
    parser.add_argument("--model", default="random_forest", choices=list(ml_core.MODELS))
    parser.add_argument("--out", type=Path, default=MODEL_PATH)
    args = parser.parse_args()

    df = load_data(args.data)
    if args.drop is None:
        drop = ml_core.suggest_drop_columns(df, args.target)
        drop += [c for c in DEFAULT_DROP if c in df.columns and c not in drop]
    else:
        drop = args.drop
    print(f"Datos: {df.shape} | objetivo: {args.target} | excluidas: {drop} | modelo: {args.model}")

    # Comparación de modelos con validación cruzada (selección sin tocar el test)
    try:
        cv_results = ml_core.cross_val_compare(df, args.target, drop, models=list(ml_core.MODELS), cv=5)
        print("\nValidación cruzada (5 folds, media ± desv):")
        for name, scores in cv_results.items():
            print(f"  {name}: f1_pos={scores['f1_pos'][0]:.4f}±{scores['f1_pos'][1]:.4f} | "
                  f"recall_pos={scores['recall_pos'][0]:.4f}±{scores['recall_pos'][1]:.4f} | "
                  f"roc_auc={scores['roc_auc'][0]:.4f}±{scores['roc_auc'][1]:.4f}")
    except Exception as exc:  # noqa: BLE001
        print(f"\n(AVISO) No se pudo calcular la validación cruzada: {exc}")

    artifact = train_model(df, args.target, drop, args.model, args.data.name)
    joblib.dump(artifact, args.out)
    assert ml_core.check_parity(joblib.load(args.out)) <= 1e-9, "Fallo de paridad tras recargar"
    print(f"Modelo guardado en {args.out}")


if __name__ == "__main__":
    main()

Reentrenar desde cero con el script (PDF §16.1): debe regenerar `model.joblib` con las mismas métricas.

In [ ]:
drop_args = " ".join(f'"{c}"' for c in drop_columns)
!cd {PROJECT} && python train.py --target "{TARGET}" --model {MODEL_NAME} --drop {drop_args}
!ls -la {PROJECT}

## 13–14. Aplicación Streamlit `app.py`
Tres pestañas:
1. **Entrenar con nuevo CSV**: botón para seleccionar cualquier CSV, elegir la columna objetivo, las columnas a excluir y el modelo. Entrenamos con el **mismo pipeline** de `ml_core.py`, mostramos las métricas y el modelo queda activo (y se puede descargar).
2. **Predecir un caso**: el formulario se genera desde el artefacto (`number_input` con mediana/rango y `selectbox` con las categorías vistas en entrenamiento, PDF §14).
3. **Predecir por lotes**: se sube un CSV, se validan las columnas (las faltantes se imputan y las sobrantes se ignoran) y se descarga el resultado.

Al arrancar, la app carga `model.joblib` **sin reentrenar** (PDF §34). Entrenar desde la app es una extensión sobre el PDF.

In [ ]:
%%writefile /content/ml_streamlit_project/app.py
"""app.py - Aplicación Streamlit (PDF §13-14 + extensión de entrenamiento con CSV nuevo).

Pestañas:
  1. Entrenar con un CSV nuevo (cualquier dataset de clasificación).
  2. Predecir un caso (formulario generado desde el artefacto).
  3. Predecir por lotes (CSV) y descargar resultados.
"""
from __future__ import annotations

import io
from pathlib import Path

import joblib
import pandas as pd
import streamlit as st

import ml_core

MODEL_PATH = Path("model.joblib")

st.set_page_config(page_title="ML App - Clasificación supervisada", layout="wide")


@st.cache_resource
def load_default_artifact():
    if MODEL_PATH.exists():
        return joblib.load(MODEL_PATH)
    return None


@st.cache_data(show_spinner=False)
def read_uploaded(data: bytes) -> pd.DataFrame:
    return ml_core.read_csv_auto(io.BytesIO(data))


if "artifact" not in st.session_state:
    st.session_state["artifact"] = load_default_artifact()
artifact = st.session_state["artifact"]

# ----------------------------------------------------------------------------
# Barra lateral: modelo activo
# ----------------------------------------------------------------------------
with st.sidebar:
    st.header("Modelo activo")
    if artifact is None:
        st.warning("No hay modelo cargado. Entrene uno en la pestaña 1.")
    else:
        st.write(f"**Dataset:** {artifact.get('dataset', '-')}")
        st.write(f"**Objetivo:** `{artifact['target']}`")
        st.write(f"**Modelo:** {ml_core.MODELS.get(artifact['model_name'], artifact['model_name'])}")
        st.write(f"**Clases:** {', '.join(map(str, artifact['classes']))}")
        st.write(f"**Variables:** {len(artifact['features'])}")
        st.json({k: round(v, 4) for k, v in artifact["metrics"].items()}, expanded=False)
        st.download_button(
            "Descargar model.joblib",
            data=(lambda b: (joblib.dump(artifact, b), b.getvalue())[1])(io.BytesIO()),
            file_name="model.joblib",
        )
        if MODEL_PATH.exists() and st.button("Restaurar modelo por defecto"):
            st.session_state["artifact"] = load_default_artifact()
            st.rerun()

st.title("Aplicación de Machine Learning supervisado")
st.caption("Pipeline scikit-learn (preprocesamiento + modelo) · Streamlit · Docker · GCP")

tab_train, tab_single, tab_batch = st.tabs(
    ["1 · Entrenar con nuevo CSV", "2 · Predecir un caso", "3 · Predecir por lotes (CSV)"]
)

# ----------------------------------------------------------------------------
# 1. Entrenar
# ----------------------------------------------------------------------------
with tab_train:
    st.subheader("Entrenar un modelo con cualquier CSV")
    st.caption("Extensión opcional: la app **no** reentrena al arrancar; esto solo ocurre si se "
               "sube un CSV y se pulsa Entrenar. En una VM pequeña, entrene datasets moderados.")
    st.write("Seleccione un archivo CSV (separador `,` o `;` detectado automáticamente).")
    up = st.file_uploader("Explorar archivos", type=["csv"], key="train_csv")
    if up is not None:
        try:
            df = read_uploaded(up.getvalue())
        except Exception as exc:  # noqa: BLE001
            st.error(f"No se pudo leer el CSV: {exc}")
            st.stop()

        c1, c2, c3 = st.columns(3)
        c1.metric("Filas", f"{df.shape[0]:,}")
        c2.metric("Columnas", df.shape[1])
        c3.metric("Celdas nulas", int(df.isna().sum().sum()))
        st.dataframe(df.head(20))
        with st.expander("Tipos y nulos por columna"):
            st.dataframe(pd.DataFrame({
                "tipo": df.dtypes.astype(str),
                "nulos": df.isna().sum(),
                "valores_distintos": df.nunique(),
            }))

        cols = df.columns.tolist()
        target = st.selectbox("Columna objetivo (y)", cols, index=len(cols) - 1)
        suggested = ml_core.suggest_drop_columns(df, target)
        drop = st.multiselect(
            "Columnas a excluir (identificadores, fuga de información...)",
            [c for c in cols if c != target], default=suggested,
        )
        cA, cB, cC = st.columns(3)
        model_name = cA.selectbox("Modelo", list(ml_core.MODELS), format_func=ml_core.MODELS.get, index=1)
        test_size = cB.slider("Proporción de prueba", 0.1, 0.4, 0.2, 0.05)
        max_rows = cC.number_input("Máximo de filas (0 = todas)", min_value=0, value=20000, step=1000,
                                   help="Limita las filas usadas para no agotar la memoria de la VM.")

        st.write("Distribución de la variable objetivo:")
        st.dataframe(df[target].value_counts(dropna=False).rename("conteo").to_frame().T)

        if st.button("Entrenar", type="primary"):
            with st.spinner("Entrenando..."):
                try:
                    new_art = ml_core.train(
                        df, target=target, drop_columns=drop, model_name=model_name,
                        test_size=test_size, max_rows=int(max_rows) or None, dataset_name=up.name,
                    )
                except Exception as exc:  # noqa: BLE001
                    st.error(str(exc))
                    st.stop()
            st.session_state["artifact"] = new_art
            artifact = new_art
            st.success("Modelo entrenado y activado. Ya puede usar las pestañas 2 y 3.")
            ev = new_art["evaluation"]
            st.write("**Métricas en el conjunto de prueba**")
            st.dataframe(pd.Series(new_art["metrics"]).round(4).to_frame("valor").T)
            st.write("**Matriz de confusión** (filas = real, columnas = predicho)")
            st.dataframe(pd.DataFrame(ev["confusion_matrix"], index=ev["labels"], columns=ev["labels"]))
            st.code(ev["classification_report"])
            st.info(f"Numéricas: {new_art['numeric_features']}\n\nCategóricas: {new_art['categorical_features']}")
            if st.button("Actualizar interfaz"):
                st.rerun()

if artifact is None:
    with tab_single:
        st.info("Primero entrene un modelo.")
    with tab_batch:
        st.info("Primero entrene un modelo.")
    st.stop()

# ----------------------------------------------------------------------------
# 2. Un caso (PDF §13-14)
# ----------------------------------------------------------------------------
with tab_single:
    st.subheader(f"Predecir `{artifact['target']}` para un nuevo caso")
    values = {}
    with st.form("single"):
        grid = st.columns(3)
        i = 0
        for feature in artifact["numeric_features"]:
            stats = artifact["numeric_stats"][feature]
            with grid[i % 3]:
                if stats["integer"]:
                    values[feature] = st.number_input(feature, value=int(round(stats["median"])), step=1,
                                                      help=f"rango entrenamiento: {stats['min']:g} – {stats['max']:g}")
                else:
                    values[feature] = st.number_input(feature, value=float(stats["median"]),
                                                      help=f"rango entrenamiento: {stats['min']:g} – {stats['max']:g}")
            i += 1
        for feature in artifact["categorical_features"]:
            with grid[i % 3]:
                values[feature] = st.selectbox(feature, artifact["category_options"][feature])
            i += 1
        submitted = st.form_submit_button("Ejecutar modelo", type="primary")
    if submitted:
        X_new = pd.DataFrame([values], columns=artifact["features"])
        out, _, _ = ml_core.predict_df(artifact, X_new)
        st.success(f"Clase predicha: **{out['prediccion'].iloc[0]}**")
        probs = out[[c for c in out.columns if c.startswith("prob_")]].T
        probs.columns = ["probabilidad"]
        st.bar_chart(probs)
        st.dataframe(probs)

# ----------------------------------------------------------------------------
# 3. Lotes
# ----------------------------------------------------------------------------
with tab_batch:
    st.subheader("Predicción por lotes")
    st.write("Suba un CSV con las columnas del modelo activo (la columna objetivo es opcional):")
    st.code(", ".join(artifact["features"]))
    template = pd.DataFrame([{f: artifact["reference"]["input"].get(f) for f in artifact["features"]}])
    st.download_button("Descargar plantilla CSV", template.to_csv(index=False).encode(), "plantilla.csv")
    up_b = st.file_uploader("Explorar archivos", type=["csv"], key="batch_csv")
    if up_b is not None:
        df_b = read_uploaded(up_b.getvalue())
        out, missing, extra = ml_core.predict_df(artifact, df_b)
        if missing:
            st.warning(f"Columnas faltantes (se imputan): {missing}")
        if extra:
            st.info(f"Columnas ignoradas: {extra}")
        st.dataframe(out)
        st.write("Distribución de predicciones:")
        st.bar_chart(out["prediccion"].astype(str).value_counts())
        if artifact["target"] in df_b.columns:
            acc = (out["prediccion"].astype(str) == df_b[artifact["target"]].astype(str)).mean()
            st.metric("Accuracy contra la columna objetivo incluida", f"{acc:.3f}")
        st.download_button("Descargar predicciones", out.to_csv(index=False).encode(), "predicciones.csv")

In [ ]:
%%writefile /content/ml_streamlit_project/.streamlit/config.toml
[server]
headless = true
maxUploadSize = 200

[browser]
gatherUsageStats = false

## 15. Probar Streamlit en Google Colab

In [ ]:
# Equivalente a: !streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > /content/streamlit.log 2>&1 &
import subprocess, time
subprocess.run(["pkill", "-f", "streamlit run"])
log = open("/content/streamlit.log", "w")
st_proc = subprocess.Popen(
    ["streamlit", "run", "/content/ml_streamlit_project/app.py", "--server.port", "8501", "--server.address", "0.0.0.0"],
    cwd="/content/ml_streamlit_project", stdout=log, stderr=subprocess.STDOUT,
)
time.sleep(10)
print(open("/content/streamlit.log").read())
!curl -s -o /dev/null -w "Health check HTTP %{http_code}\n" http://localhost:8501/_stcore/health

Túnel temporal para verlo desde el navegador (solo para pruebas, **no** es el despliegue final).
La contraseña que pide localtunnel es la IP pública de Colab que imprime la primera línea.
Si la red bloquea el túnel, basta con que el log anterior muestre que Streamlit inició.
Detenga la celda (■) para cerrar el túnel.

In [ ]:
!curl -s https://loca.lt/mytunnelpassword; echo
!npx --yes localtunnel --port 8501

## 16. Pruebas mínimas antes de Docker

In [ ]:
import numpy as np

loaded = joblib.load(PROJECT / "model.joblib")

# 1) Paridad: el artefacto recargado reproduce la predicción de referencia
diff = ml_core.check_parity(loaded)
assert diff <= 1e-9, diff
print("Paridad verificada, diferencia máxima:", diff)

# 2) Las columnas de entrada coinciden con las de entrenamiento
assert loaded["features"] == [c for c in df.columns if c not in drop_columns + [TARGET]]

# 3) Tres casos distintos + valores extremos + categoría inválida + columna faltante/sobrante
casos = df.drop(columns=[TARGET]).sample(3, random_state=1).copy()
extremo = casos.iloc[[0]].copy()
for c in loaded["numeric_features"]:
    extremo[c] = loaded["numeric_stats"][c]["max"] * 10
if loaded["categorical_features"]:
    extremo[loaded["categorical_features"][0]] = "categoria_nunca_vista"
lote = pd.concat([casos, extremo]).drop(columns=[loaded["features"][0]])
lote["columna_extra"] = 1
lote.iloc[1, 0] = np.nan
out, missing, extra = ml_core.predict_df(loaded, lote)
print("Faltantes:", missing, "| Ignoradas:", extra)
display(out.iloc[:, -(len(loaded["classes"]) + 1):])

# 4) Entrenar con otro dataset del repo (lo mismo que hace la pestaña 1 de la app)
other = ml_core.read_csv_auto("https://raw.githubusercontent.com/wilmerpineda/machine_learning_ii_course_public/"
                              "main/datasets/public/Default_Clientes.csv")
art2 = ml_core.train(other, "Default", ml_core.suggest_drop_columns(other, "Default"), "logistic_regression")
print("Default_Clientes ->", {k: round(v, 4) for k, v in art2["metrics"].items()})

# 5) Ejemplo de entrada/salida para el README
ejemplo = pd.DataFrame([loaded["reference"]["input"]])
EJEMPLO_OUT, _, _ = ml_core.predict_df(loaded, ejemplo)
display(EJEMPLO_OUT)

## 17. Archivos para Docker
`requirements.txt` se escribe con **las versiones exactas instaladas en este Colab**: el `model.joblib`
debe cargarse en Docker con la misma versión de scikit-learn con la que se entrenó.

In [ ]:
from importlib.metadata import version
pkgs = ["streamlit", "pandas", "numpy", "scikit-learn", "joblib", "openpyxl"]
reqs = "\n".join(f"{p}=={version(p)}" for p in pkgs) + "\n"
(PROJECT / "requirements.txt").write_text(reqs)
print(reqs)
print("Python de Colab:", sys.version.split()[0])

In [ ]:
%%writefile /content/ml_streamlit_project/.dockerignore
.git
.venv
__pycache__
*.pyc
.ipynb_checkpoints
.env
*.log
*.zip
*.pdf
pdf/
docs/
*.ipynb
src/

In [ ]:
%%writefile /content/ml_streamlit_project/Dockerfile
FROM python:3.12-slim

WORKDIR /app

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8501

HEALTHCHECK --interval=30s --timeout=5s --start-period=20s --retries=3 \
    CMD python -c "import urllib.request; urllib.request.urlopen('http://127.0.0.1:8501/_stcore/health', timeout=3)"

CMD ["streamlit", "run", "app.py", \
     "--server.address=0.0.0.0", \
     "--server.port=8501"]

## 18. ¿Qué significa cada instrucción Docker?
| Instrucción | Significado |
|---|---|
| `FROM python:3.12-slim` | Imagen base pequeña con Python 3.12 |
| `WORKDIR /app` | Directorio de trabajo dentro de la imagen |
| `COPY requirements.txt .` | Copia primero las dependencias para aprovechar la caché de capas |
| `RUN pip install ...` | Instala las librerías dentro de la imagen |
| `COPY . .` | Copia el proyecto (app.py, ml_core.py, model.joblib, data/, .streamlit/) |
| `EXPOSE 8501` | Documenta el puerto de Streamlit |
| `CMD ...` | Proceso que se ejecuta al iniciar el contenedor |

El Dockerfile y el `.dockerignore` se embeben tal cual están en el repositorio (una sola fuente de verdad);
construimos la imagen en la VM de GCP.

## 20. README completo

El `README.md` del repositorio se embebe tal cual (una sola fuente de verdad):

In [ ]:
%%writefile /content/ml_streamlit_project/README.md
# Despliegue de Modelos de Machine Learning: Pipeline Supervisado, Contenedorización con Docker y Servicio en Google Cloud Platform

**Asignatura:** Machine Learning II  
**Programa:** Pregrado en Ciencia de Datos  
**Modalidad:** Reto Práctico de Ciclo de Vida Completo de Machine Learning  

---

## 1. Descripción y Propósito del Proyecto

En este proyecto implementamos el ciclo de vida completo de una solución de Machine Learning aplicada: desde la ingesta de datos y el preprocesamiento automatizado hasta el entrenamiento, la evaluación estadística, la serialización de artefactos, el desarrollo de la interfaz interactiva, la contenedorización con Docker y el despliegue en una máquina virtual de infraestructura como servicio (IaaS) en Google Cloud Platform (Compute Engine).

Nuestro objetivo principal es garantizar la reproducibilidad estricta del sistema en entornos heterogéneos (Google Colab, entornos locales basados en contenedores y servidores de producción en la nube), asegurar la paridad numérica en inferencia y desacoplar la fase de entrenamiento de la fase de servicio.

---

## 2. Conjunto de Datos y Unidad de Observación

- **Fuente:** Repositorio académico oficial del curso ([`Customer_Churn_Iran.csv`](https://raw.githubusercontent.com/wilmerpineda/machine_learning_ii_course_public/main/datasets/public/Customer_Churn_Iran.csv)).
- **Unidad de Análisis:** Cada registro corresponde a un cliente individual de una empresa de telecomunicaciones, evaluado a lo largo de un período de observación de 12 meses.
- **Volumen:** 3.150 observaciones y 14 columnas tabulares sin presencia de valores nulos estructurales.
- **Variable Objetivo (`Target`):** `Churn` (variable binaria, donde `1` representa el abandono del servicio por parte del usuario y `0` representa la retención).
- **Distribución de Clases:** Trabajamos con un problema de clases desbalanceadas (aproximadamente 15,7% de casos positivos y 84,3% de casos negativos), lo que nos exige una estrategia de evaluación especializada.

---

## 3. Ruta Metodológica

Desarrollamos el proyecto bajo la **Ruta A (Aprendizaje Supervisado - Clasificación Binaria)** definida en la guía de laboratorio:

- Modelamiento predictivo supervisado con partición estratificada de datos (80% entrenamiento, 20% prueba, `random_state=42`).
- Arquitectura basada en `Pipeline` de `scikit-learn` que integra de forma atómica la imputación, estandarización, codificación categórica y estimador.
- Serialización de artefactos con metadatos completos y caso de referencia para comprobación de paridad numérica.

---

## 4. Definición y Justificación de Variables

A fin de evitar fuga de información (*data leakage*) y asegurar que la interfaz de usuario reciba únicamente información disponible al momento de la toma de decisiones, aplicamos el siguiente esquema de variables. La variable objetivo es `Churn` (1 = abandono, 0 = permanencia), con un desbalance aproximado de 15,7 % positivos / 84,3 % negativos; por eso la partición es estratificada y la evaluación prioriza la clase positiva.

### 4.1 Variables Predictoras Incluidas

| Variable | Tipo | Por qué se incluye | Tratamiento en el pipeline |
|:---|:---|:---|:---|
| `Call Failure` | Numérica discreta (conteo) | Frecuencia de llamadas fallidas; mide problemas de red o insatisfacción y es una señal temprana de abandono. | Mediana + estandarización |
| `Subscription Length` | Numérica (meses) | Antigüedad del cliente; distingue clientes nuevos de clientes con relación establecida. | Mediana + estandarización |
| `Charge Amount` | Numérica discreta (nivel tarifario) | Nivel de cobro o plan; aproxima el valor económico del cliente. | Mediana + estandarización |
| `Seconds of Use` | Numérica continua (segundos) | Intensidad de consumo; un mayor uso sugiere dependencia del servicio. | Mediana + estandarización |
| `Frequency of use` | Numérica discreta (conteo) | Número de llamadas efectuadas; intensidad de la relación con la operadora. | Mediana + estandarización |
| `Frequency of SMS` | Numérica discreta (conteo) | Uso de mensajes de texto; nivel de *engagement* del cliente. | Mediana + estandarización |
| `Distinct Called Numbers` | Numérica discreta (conteo) | Diversidad de contactos; tamaño de la red social del cliente. | Mediana + estandarización |
| `Complains` | Categórica binaria (0/1) | Presencia de queja formal; predictor directo de insatisfacción y churn. | Moda + one-hot |
| `Age Group` | Categórica ordinal (1–5) | Segmento etario; captura heterogeneidad demográfica en la propensión al abandono. | Moda + one-hot (no se impone orden) |
| `Tariff Plan` | Categórica nominal (1 = Contractual, 2 = Prepago) | Tipo de plan; el prepago suele asociarse a mayor rotación. | Moda + one-hot |
| `Status` | Categórica nominal (1, 2) | Estado operativo de la línea; puede reflejar actividad o suspensión del servicio. | Moda + one-hot |

### 4.2 Variables Excluidas

| Variable | Motivo de exclusión |
|:---|:---|
| `Customer Value` | Se calcula *a posteriori* a partir de métricas agregadas de uso. Incluirla generaría fuga de información y no está disponible para un cliente nuevo o en una intervención temprana. |
| `Age` | Presenta una correspondencia **1:1** con `Age Group` (15→1, 25→2, 30→3, 45→4, 55→5). Mantener ambas duplicaría exactamente las mismas columnas one-hot sin aportar información; conservamos `Age Group`, que es el agrupamiento interpretable. |
| `Churn` | Es la variable objetivo; no entra como predictor. |

> No incluimos identificadores porque el dataset no los contiene. Si existieran columnas como `ID` o `Customer ID`, las excluiríamos para evitar que el modelo memorice clientes en lugar de aprender patrones.

---

## 5. Preparación de Datos y Arquitectura del Pipeline

Para asegurar que las mismas transformaciones del entrenamiento se repliquen exactamente durante la inferencia, utilizamos un `ColumnTransformer` encapsulado en un `Pipeline`:

```
Entrada tabular (CSV o Formulario)
       │
       ├── Columnas Numéricas ──────> SimpleImputer(median) ─────────> StandardScaler() ───┐
       │                                                                                   │
       └── Columnas Categóricas ────> SimpleImputer(most_frequent) ──> OneHotEncoder() ────┤
                                                                                           │
                                                                   Concatena matriz de features
                                                                                           │
                                                                                           ▼
                                                                             Estimador (Clasificador)
```

- **Manejo de categorías no observadas:** El codificador `OneHotEncoder` utiliza el parámetro `handle_unknown="ignore"`, asignando un vector nulo a categorías no presentes en la fase de entrenamiento, evitando fallas críticas en tiempo de ejecución.
- **Normalización de identificadores de columna:** Procesamos los encabezados para eliminar codificaciones BOM y duplicidad de espacios tipográficos presentes en el archivo fuente.

---

## 6. Selección y Evaluación del Modelo

Realizamos una comparativa empírica entre el modelo base propuesto en la guía (`LogisticRegression`) y un ensamble de árboles (`RandomForestClassifier`, configurado con 300 estimadores, `min_samples_leaf=5`, `class_weight="balanced"` y `random_state=42`).

### 6.1 Selección con Validación Cruzada (5-fold estratificada)

Para evitar el sesgo de elegir "el mejor" modelo sobre el propio conjunto de prueba, hicimos la selección con **validación cruzada estratificada de 5 folds** sobre los datos de entrenamiento, reportando la clase positiva (`Churn=1`):

| Métrica (media ± desv., CV) | Regresión Logística (Base) | Random Forest (Seleccionado) |
|:---|:---:|:---:|
| **F1 (Clase Churn = 1)** | 0.5672 ± 0.0217 | **0.8406 ± 0.0226** |
| **Recall (Clase Churn = 1)** | 0.4404 ± 0.0290 | **0.9414 ± 0.0234** |
| **ROC-AUC** | 0.9354 ± 0.0071 | **0.9824 ± 0.0046** |

El **Random Forest** dominó consistentemente las tres métricas, por lo que lo seleccionamos. Reservamos el conjunto de prueba exclusivamente para la evaluación final (sección 6.2), sin usarlo en la selección.

### 6.2 Matriz de Confusión (Random Forest en Conjunto de Prueba, N = 630)

| Condición Real \ Predicción | Predicho Negativo (0) | Predicho Positivo (1) | Total Real |
|:---|:---:|:---:|:---:|
| **Real Negativo (0)** | 505 | 26 | 531 |
| **Real Positivo (1)** | 4 | 95 | 99 |

### 6.3 Métricas Cuantitativas Comparativas (Conjunto de Prueba)

| Métrica | Regresión Logística (Base) | Random Forest (Seleccionado) |
|:---|:---:|:---:|
| **Accuracy Global** | 0.8968 | **0.9524** |
| **Balanced Accuracy** | 0.7046 | **0.9553** |
| **Recall (Clase Churn = 1)** | 0.4242 | **0.9596** |
| **Precision (Clase Churn = 1)** | 0.8400 | **0.7851** |
| **F1 (Clase Churn = 1)** | 0.5638 | **0.8636** |
| **F1-Score Ponderado** | 0.8821 | **0.9543** |
| **ROC-AUC** | 0.9252 | **0.9844** |
| **Precision-Recall (Average Precision)** | 0.7616 | **0.9353** |

### 6.4 Justificación Técnica de la Elección
Debido a la asimetría en los costos de error asociados al problema de negocio, no identificar a un cliente que efectivamente abandonará el servicio (falso negativo) conlleva la pérdida del valor de vida del cliente (*lifetime value*), mientras que un falso positivo implica únicamente una acción preventiva de contacto. Por consiguiente, seleccionamos el modelo **Random Forest** por su capacidad de captura (**Recall de 95,96%** frente a 42,42% del modelo lineal) y su **F1 de la clase positiva (0.8636 frente a 0.5638)**. Reportamos el F1 de la clase positiva (no solo el ponderado, dominado por la clase mayoritaria) junto con la precisión-recall (*Average Precision*) para reflejar el desempeño sobre la clase de interés.

---

## 7. Estructura del Repositorio

Mantenemos una organización modular y desacoplada:

```
├── .streamlit/
│   └── config.toml          # Parámetros del servidor Streamlit (headless, maxUploadSize)
├── data/
│   └── datos.csv            # Copia local del dataset Customer_Churn_Iran.csv
├── pdf/
│   └── guia_completa_ml_streamlit_docker_gcp_colab.pdf # Guía académica del taller
├── src/
│   └── build_notebook.py    # Generador automatizado del cuaderno unificado
├── .dockerignore            # Reglas de exclusión para el contexto de Docker
├── .gitignore               # Reglas de exclusión para Git
├── app.py                   # Aplicación interactiva de servicio Streamlit
├── Dockerfile               # Instrucciones de construcción de la imagen de producción
├── ML_Supervisado_Streamlit_Docker_GCP.ipynb # Cuaderno ejecutable en Google Colab
├── ml_core.py               # Módulo central: inferencia de tipos, pipelines, métricas y CV
├── model.joblib             # Artefacto serializado (modelo entrenado + metadatos)
├── README.md                # Documentación técnica institucional
├── requirements.txt         # Dependencias con versiones fijadas (reproducibilidad)
└── train.py                 # Script de entrenamiento por línea de comandos
```

---

## 8. Instrucciones de Reproducción y Ejecución

### 8.1 Ejecución Completa desde Google Colab
1. Cargar el archivo `ML_Supervisado_Streamlit_Docker_GCP.ipynb` en Google Colab.
2. Ejecutar la secuencia a través del comando `Entorno de ejecución` -> `Ejecutar todo`.
3. El cuaderno genera de manera autónoma los scripts modulares, el artefacto entrenado, las pruebas de paridad y descarga el paquete `ml_streamlit_project.zip`.

### 8.2 Ejecución Local en Entorno Virtual

```bash
# 1. Creación e inicialización del entorno virtual
python3 -m venv .venv
source .venv/bin/activate

# 2. Instalación de dependencias del proyecto
pip install --upgrade pip
pip install -r requirements.txt

# 3. Ejecución del script de entrenamiento (reconstrucción de model.joblib)
python train.py

# 4. Despliegue local de la aplicación interactiva
streamlit run app.py --server.port=8501 --server.address=0.0.0.0
```

Acceder a través del navegador web a `http://localhost:8501`.

> **Nota sobre la pestaña de entrenamiento.** La aplicación no reentrena el modelo al arrancar: siempre sirve `model.joblib`. Incluye, como **extensión opcional**, una pestaña para entrenar con un CSV nuevo (solo si el usuario lo sube y pulsa "Entrenar"). Para no agotar la memoria de la VM, el número de filas está limitado por defecto a 20.000.

### 8.3 Captura de la Aplicación en Ejecución

![Predicción de un caso en Streamlit](docs/evidencias/streamlit-prediccion.png)

![Resultado de la predicción](docs/evidencias/streamlit-resultado.png)

Las capturas corresponden al contenedor `ml-streamlit` corriendo en `http://localhost:8501`. La predicción de ejemplo devolvió la clase `0` y las probabilidades que se ven en la gráfica. La pestaña de entrenamiento está en [`docs/evidencias/streamlit-entrenamiento.png`](docs/evidencias/streamlit-entrenamiento.png).

---

## 9. Empaquetamiento y Contenedorización con Docker

Empaquetamos el sistema en una imagen ligera basada en `python:3.12-slim`. La aplicación no requiere privilegios elevados y expone el puerto estándar `8501`.

### 9.1 Construcción de la Imagen Docker

```bash
docker build -t ml-streamlit-app .
```

### 9.2 Ejecución y Verificación del Contenedor

```bash
# Ejecución interactiva para verificación de logs de arranque
docker run --rm -p 8501:8501 ml-streamlit-app

# Ejecución en segundo plano con política de reinicio automático
docker run -d \
  --name ml-streamlit \
  --restart unless-stopped \
  -p 8501:8501 \
  ml-streamlit-app

# Inspección de estado y diagnóstico
docker ps
docker logs ml-streamlit
```

### 9.3 Evidencia de Construcción y Ejecución (verificada)

Registramos la evidencia el **2026-11-10** en un entorno Linux/WSL con Docker 29.1.3:

| Verificación | Resultado |
|:---|:---|
| `docker build -t ml-streamlit-app .` | `Successfully tagged ml-streamlit-app:latest` |
| `docker images ml-streamlit-app` | Imagen creada (≈245 MB de contenido) |
| `docker ps --filter name=ml-streamlit` | `Up (healthy)`, puerto `0.0.0.0:8501->8501/tcp` |
| `curl http://127.0.0.1:8501/_stcore/health` | `ok` |
| Carga del artefacto dentro del contenedor | `sklearn_version_artifact: 1.9.1`, sin `InconsistentVersionWarning` |
| Paridad numérica del caso de referencia | `parity_diff: 0.0` |
| Predicción de ejemplo dentro del contenedor | Clase `0`, `P(Y=0)=0.5093`, `P(Y=1)=0.4907` |

Guardamos los logs completos en:

- [`docs/evidencias/docker-build.log`](docs/evidencias/docker-build.log)
- [`docs/evidencias/docker-images.log`](docs/evidencias/docker-images.log)
- [`docs/evidencias/docker-run.log`](docs/evidencias/docker-run.log) (incluye `docker ps`, `docker logs` y healthcheck)
- [`docs/evidencias/docker-model-check.log`](docs/evidencias/docker-model-check.log)

Para reproducir la verificación:

```bash
docker build -t ml-streamlit-app .
docker run -d --name ml-streamlit --restart unless-stopped -p 8501:8501 ml-streamlit-app
sleep 10
docker ps
docker logs ml-streamlit
curl http://127.0.0.1:8501/_stcore/health
```

### 9.4 Evidencia de Entrenamiento y Prueba Funcional dentro del Contenedor

Ejecutamos el entrenamiento completo dentro de la imagen con `docker run --rm ml-streamlit-app python train.py`. El script regeneró `model.joblib` con las versiones fijadas, calculó la validación cruzada y verificó la paridad al recargar:

| Verificación | Resultado |
|:---|:---|
| Entrenamiento dentro de la imagen | `Modelo guardado en model.joblib` |
| Validación cruzada (5-fold) | Random Forest: F1 clase 1 `0.8406`, Recall `0.9414`, ROC-AUC `0.9824` |
| Métricas en test | Accuracy `0.9524`, F1 clase 1 `0.8636`, ROC-AUC `0.9844` |
| Paridad tras recargar | verificada por el `assert` de `train.py` |
| Prueba funcional de Streamlit (`AppTest`) | sin excepciones; 7 entradas numéricas, 4 selectores y `Clase predicha: 0` |

Logs:

- [`docs/evidencias/train-in-container.log`](docs/evidencias/train-in-container.log)
- [`docs/evidencias/streamlit-apptest.log`](docs/evidencias/streamlit-apptest.log)

---

## 10. Despliegue en Google Cloud Platform (Compute Engine)

A continuación detallamos el protocolo para desplegar la aplicación en una instancia virtual de Google Cloud:

### 10.1 Creación de la Instancia Virtual
1. Acceder a la consola de administración de Google Cloud Platform (`console.cloud.google.com`).
2. Navegar a **Compute Engine** -> **Instancias de VM**.
3. Seleccionar **Crear instancia** con las siguientes especificaciones:
   - **Sistema operativo:** Ubuntu 24.04 LTS (x86/64).
   - **Tipo de máquina:** `e2-small` o `e2-medium` (recomendado para estabilidad de memoria).
   - **IP Externa:** Asignación efímera o estática habilitada.
   - **Etiqueta de red:** `streamlit-server`.

### 10.2 Configuración del Cortafuegos (Firewall)
Para permitir el tráfico entrante al puerto 8501, ejecutar en Google Cloud Shell o crear la regla vía interfaz de red VPC:

```bash
gcloud compute firewall-rules create allow-streamlit-8501 \
    --allow=tcp:8501 \
    --direction=INGRESS \
    --target-tags=streamlit-server \
    --description="Permitir trafico entrante TCP en puerto 8501 para Streamlit"
```

### 10.3 Instalación, Clonación y Puesta en Marcha en la VM
Conectarse mediante la terminal SSH integrada en la consola de Compute Engine y ejecutar:

```bash
# 1. Actualización e instalación del motor Docker y Git
sudo apt update && sudo apt install -y docker.io git
sudo systemctl enable --now docker

# 2. Clonación del repositorio institucional
git clone https://github.com/julianjimenez4809-ui/MLII_Despliegue_Streamlit_Docker_GCP.git
cd MLII_Despliegue_Streamlit_Docker_GCP

# 3. Construcción y ejecución del contenedor en el servidor
sudo docker build -t ml-streamlit-app .
sudo docker run -d --name ml-streamlit --restart unless-stopped -p 8501:8501 ml-streamlit-app
```

### 10.4 Verificación de Servicio
Abrir un navegador web e ingresar la dirección IP externa de la máquina virtual:

```
http://<IP_EXTERNA_DE_LA_VM>:8501
```

### 10.5 Estado de Verificación en GCP

**No verificada.** No ejecutamos el despliegue en Google Cloud porque la creación de la cuenta de facturación de GCP exige un prepago o cargo de verificación que no se reembolsa de forma inmediata, y decidimos no asumirlo. Esta sección queda como protocolo reproducible, no como evidencia ejecutada. La verificación funcional del contenedor la realizamos localmente y está documentada en la sección 9.3.

---

## 11. Casos de Prueba y Contratos de Inferencia

### 11.1 Ejemplo de Entrada (Registro de Prueba)
```json
{
  "Call Failure": 2,
  "Complains": "0",
  "Subscription Length": 36,
  "Charge Amount": 0,
  "Seconds of Use": 1260,
  "Frequency of use": 39,
  "Frequency of SMS": 20,
  "Distinct Called Numbers": 22,
  "Age Group": "3",
  "Tariff Plan": "1",
  "Status": "2"
}
```

### 11.2 Salida Generada por el Modelo
- **Clase Predicha:** `0` (Permanencia / No Abandono).
- **Probabilidades Estimadas:** 
  - Probabilidad de Permanencia (`P(Y=0)`): 0.5093
  - Probabilidad de Abandono (`P(Y=1)`): 0.4907

---

## 12. Protocolo de Auditoría y Prueba Cruzada

Para validar la reproducibilidad externa requerida en la sección 31 de la guía, pedimos a un evaluador independiente que opere el sistema siguiendo exclusivamente los siguientes pasos:

1. Clonar el repositorio mediante la URL pública.
2. Comprobar la integridad de los archivos raíz (`Dockerfile`, `requirements.txt`, `app.py`, `ml_core.py`, `model.joblib`).
3. Ejecutar el comando `sudo docker build -t ml-streamlit-app .` sin alertas de falla de dependencias.
4. Ejecutar el contenedor mediante `sudo docker run -d --name ml-streamlit -p 8501:8501 ml-streamlit-app`.
5. Verificar el estado con `sudo docker ps` y revisar los logs con `sudo docker logs ml-streamlit`.
6. Enviar una petición a través de la pestaña de predicción unitaria o predicción por lotes y verificar la respuesta esperada.

> Estos pasos incorporan dos observaciones de la prueba cruzada de la sección 12.1: incluir `ml_core.py` en la verificación de integridad y asignar `--name ml-streamlit` para que el comando de logs funcione directamente.

> **Nota sobre permisos.** Los pasos 3 y 4 usan `sudo` porque el usuario que sigue el protocolo en la VM puede no pertenecer aún al grupo `docker`. Quien haya ejecutado `sudo usermod -aG docker $USER` (y reiniciado la sesión SSH) puede omitir `sudo`.

### 12.1 Evidencia de la Prueba Cruzada (PDF §31)

| Campo | Detalle |
|:---|:---|
| **Pareja evaluadora** | Ailyn Gómez |
| **Fecha** | 2026-11-10 |
| **Entorno** | Ubuntu 24.04 (WSL2), Docker 29.1.3, imagen base `python:3.12-slim` |
| **Commit evaluado** | `b0d5b65` |
| **Pasos ejecutados** | Clonar el repositorio → leer únicamente el README → `docker build -t ml-streamlit-crosstest .` → `docker run -d --name ml-streamlit-crosstest -p 8502:8501 ml-streamlit-crosstest` → verificar healthcheck → probar un caso con el artefacto |
| **Resultado** | **Exitoso con observaciones menores.** Construimos la imagen sin errores, el contenedor quedó `Up (healthy)`, `/_stcore/health` respondió `ok` y la predicción del caso de referencia devolvió clase `0` con `P(Y=1)=0.4907`. |
| **Errores o instrucciones faltantes** | 1) El paso 2 no menciona `ml_core.py` en la verificación de integridad, aunque la app lo importa; lo verificamos manualmente y estaba presente. 2) El paso 4 no asigna `--name`, así que el `docker logs ml-streamlit` del diagnóstico no aplica directamente; obtuvimos el nombre con `docker ps`. 3) La sección 8.2 asume `python3-venv`; en el entorno de prueba no estaba instalado, pero el flujo Docker no lo requiere. |
| **Evidencia** | [`docs/evidencias/cross-test/cross-build.log`](docs/evidencias/cross-test/cross-build.log) y [`docs/evidencias/cross-test/cross-run.log`](docs/evidencias/cross-test/cross-run.log) |

### 12.2 Evidencia del Despliegue en la VM (GCP)

| Campo | Detalle |
|:---|:---|
| **URL / IP externa** | No aplica: no realizamos el despliegue |
| **Fecha del despliegue** | No realizada |
| **Instancia** | No creada |
| **Estado del contenedor** | Verificado localmente (sección 9.3), no en GCP |
| **Captura de la aplicación** | No disponible para GCP |
| **Motivo** | GCP exige un prepago o cargo de verificación que no se reembolsa de forma inmediata; decidimos no asumirlo. |

---

## 13. Desaprovisionamiento y Control de Costos

Al finalizar la demostración evaluativa o sesión de auditoría, debemos detener o desaprovisionar los recursos de cómputo para evitar facturación recurrente en Google Cloud:

```bash
# Detención de la máquina virtual vía gcloud CLI
gcloud compute instances stop <NOMBRE_DE_LA_VM> --zone=<ZONA>

# O eliminación definitiva de la instancia y sus discos asociados
gcloud compute instances delete <NOMBRE_DE_LA_VM> --zone=<ZONA>
```

---

## 14. Autores

- **Julián Jiménez**  
  *Programa de Pregrado en Ciencia de Datos*  
  *Repositorio Académico:* [julianjimenez4809-ui](https://github.com/julianjimenez4809-ui)

- **Tomás Rincón**  
  *Programa de Pregrado en Ciencia de Datos*

---

## 15. Estado de Cumplimiento

| Componente | Estado | Evidencia |
|:---|:---|:---|
| Datos y variables | Completo | Secciones 2 y 4 |
| Modelamiento y métricas | Completo | Sección 6 |
| Streamlit | Completo (local) | Sección 8.3 y capturas |
| Docker build/run | Completo | Secciones 9.3 y 9.4, `docs/evidencias/` |
| GCP | No ejecutado | Sección 10.5 |
| README | Completo | Este documento |
| Prueba cruzada | Completa | Sección 12.1 |

> La prueba cruzada la realizamos sobre el commit `b0d5b65`; los cambios posteriores de redacción no alteran el flujo técnico verificado.

---

## 16. Diagnóstico de Problemas Frecuentes

| Situación | Revisión |
|:---|:---|
| `train.py` falla | Verificar la ruta de `data/datos.csv`, el nombre de la columna objetivo y que las dependencias de `requirements.txt` estén instaladas. |
| `model.joblib` no existe | Confirmar que `joblib.dump` se ejecutó y que la ruta de salida es la raíz del proyecto. |
| El contenedor se detiene | Revisar `docker ps -a` y `docker logs ml-streamlit`. |
| El puerto no responde | Confirmar `-p 8501:8501` y que Streamlit escucha en `0.0.0.0`. |
| La app abre en la VM pero no desde fuera | Revisar la regla de firewall TCP 8501, la etiqueta `streamlit-server` y la IP externa. |
| Error al cargar `model.joblib` | Verificar que las versiones de `requirements.txt` coinciden con las del entrenamiento (`scikit-learn==1.9.1`). |
| Los cambios no aparecen | `git pull` → detener/eliminar el contenedor → `docker build` → `docker run`. |

---

## 17. Checklist Final de Entrega

- [x] Entendemos qué representa una fila del dataset.
- [x] Definimos la ruta supervisada.
- [x] Justificamos las variables una por una.
- [x] El preprocesamiento está dentro del pipeline.
- [x] `train.py` corre de principio a fin y regenera `model.joblib` (sección 9.4).
- [x] `app.py` no reentrena el modelo al arrancar.
- [x] Probamos Streamlit y la predicción unitaria (sección 8.3).
- [x] Creamos `requirements.txt`, `Dockerfile` y `.dockerignore`.
- [x] El repositorio contiene todos los archivos necesarios.
- [x] `docker build` y `docker run` funcionan y el contenedor queda healthy (sección 9.3).
- [x] `docker logs` no muestra errores críticos.
- [x] La prueba cruzada quedó documentada (sección 12.1).
- [x] Sabemos detener o eliminar la VM para controlar costos (sección 13).
- [ ] La VM de GCP está creada y accesible por SSH.
- [ ] La regla de firewall está configurada.
- [ ] La aplicación abre usando la IP externa.

> Los tres ítems pendientes corresponden a GCP. No los ejecutamos porque la cuenta de facturación exige un prepago o cargo de verificación que no se reembolsa de forma inmediata (sección 10.5).

## 21. Empaquetar y descargar el proyecto (para subirlo a GitHub)

In [ ]:
!cd /content && rm -f ml_streamlit_project.zip && zip -qr ml_streamlit_project.zip ml_streamlit_project -x "*/__pycache__/*"
!ls -la /content/ml_streamlit_project /content/ml_streamlit_project.zip
try:
    from google.colab import files
    files.download("/content/ml_streamlit_project.zip")
except ImportError:
    print("Fuera de Colab: el zip está en /content/ml_streamlit_project.zip")

## 22–31. Despliegue en GCP y prueba cruzada (pasos manuales)
1. Crear un repositorio en GitHub y subir el **contenido descomprimido** del zip, incluido este notebook. No subir credenciales.
2. Seguir la sección *Despliegue en Google Cloud* del `README.md` (VM Ubuntu 24.04 → Docker → `git clone` → `docker build` → `docker run` → firewall → `http://IP:8501`).
3. **Prueba cruzada (PDF §31):** otra pareja recibe solo la URL del repositorio, sigue únicamente el README, construye, ejecuta, prueba un caso y registra los problemas que encuentre.
4. Detener o eliminar la VM al finalizar.

## 34. Checklist final
- [ ] Entendemos qué representa una fila · [ ] Ruta supervisada definida · [ ] Variables justificadas
- [ ] Preprocesamiento dentro del pipeline · [ ] `train.py` corre de principio a fin · [ ] `model.joblib` se genera
- [ ] `app.py` no reentrena al arrancar · [ ] Probamos Streamlit en Colab · [ ] requirements/Dockerfile/.dockerignore
- [ ] Repositorio completo · [ ] VM con SSH · [ ] Docker instalado · [ ] `docker build` OK · [ ] `docker run` activo
- [ ] `docker logs` sin errores · [ ] Firewall · [ ] App abre con la IP externa · [ ] Prueba cruzada · [ ] VM detenida/eliminada